# 🩻 PneumoLens — Phase 4
## Complete leakage-aware similarity screening & candidate split approval

**Pediatric chest X-ray classification · Research/education only · Kaggle CPU notebook · GitHub Phase 4**


### 1. Environment and folders
The notebook is self-contained: it exports the exact `src/data/split.py`, config, tests, documentation, and final private manifests used in the run. No GPU or network installation is required on a standard Kaggle Python environment.

In [1]:
from __future__ import annotations
import hashlib, io, json, os, platform, subprocess, sys, textwrap, zipfile, shutil
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import FileLink, display
import sklearn

WORK_ROOT = Path(os.environ.get("PNEUMOLENS_PHASE4_V2_WORKDIR", "/kaggle/working/pneumolens_phase4_v2"))
PROJECT_ROOT = WORK_ROOT / "project_files"
# Clear only this notebook-owned generated tree on repeat runs.
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
for folder in ["src/data", "configs", "tests", "docs", "data/processed",
               "reports/Phase 4", "reports/tables", "reports/figures"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)
for pkg in ["src/__init__.py", "src/data/__init__.py"]:
    (PROJECT_ROOT / pkg).write_text('"""PneumoLens reproducible research modules."""\n')
sys.path.insert(0, str(PROJECT_ROOT))
RUNTIME = {"python": platform.python_version(), "numpy": np.__version__,
           "pandas": pd.__version__, "pillow": Image.__version__,
           "scikit_learn": sklearn.__version__}
print("Working directory:", WORK_ROOT)
print("Runtime:", RUNTIME)

Working directory: /kaggle/working/pneumolens_phase4_v2
Runtime: {'python': '3.13.16', 'numpy': '2.1.3', 'pandas': '2.3.3', 'pillow': '13.0.0', 'scikit_learn': '1.6.1'}


### 2. Freeze the Phase 4 V2 policy
This policy preserves the original source fingerprint, 624-image locked test, 20% candidate validation target, and the *predeclared* pHash and structural-metric thresholds from V1. The V2 change is **complete coverage** rather than a truncated candidate search.

In [2]:
POLICY_TEXT = r'''
{
  "version": "pneumolens-split-v2",
  "project": "PneumoLens",
  "github_phase": "Phase 4 - Complete bounded similarity screening and split integrity",
  "scope": "pediatric_binary_normal_vs_pneumonia_research_only",
  "source": {
    "expected_counts": {
      "train/NORMAL": 1349,
      "train/PNEUMONIA": 3883,
      "test/NORMAL": 234,
      "test/PNEUMONIA": 390
    },
    "strict_count_verification": true,
    "original_test_locked": true,
    "expected_dataset_file_sha256_fingerprint": "a7b36506fbccf5f77b5d80a73b7a53cca8515dba45ae959a8d0a62e5f219a87b",
    "strict_inventory_fingerprint": true,
    "reference_fingerprint_method": "SHA256 of sorted (relative_path|raw_file_sha256) lines, as in Phase1 manifest V2"
  },
  "grouping": {
    "rule": "Union of prefixed filename-series CANDIDATES, SHA256 file/decoded exact duplicates, and strong within-original-train perceptual candidates",
    "patient_ids_verified": false,
    "numeric_token_alone_is_not_patient_id": true,
    "train_deduplication": "remove redundant decoded-pixel-exact copies from original train only; maintain all original test rows",
    "no_automatic_pneumonia_subtype_relabelling": true
  },
  "validation": {
    "method": "StratifiedGroupKFold selected on original train only",
    "n_splits": 5,
    "candidate_seeds": [
      2026,
      2027,
      2031
    ],
    "target_fraction": 0.2,
    "max_abs_fraction_deviation": 0.06,
    "max_abs_positive_rate_deviation": 0.06
  },
  "near_duplicate_screen": {
    "method": "Exhaustive 64bit pHash Hamming screening, then actual 96px image-structure comparison",
    "max_phash_distance": 6,
    "chunk_size": 128,
    "strong_phash_max": 3,
    "strong_corr_min": 0.995,
    "strong_gradient_min": 0.975,
    "strong_mae_max": 0.045,
    "strong_aspect_max": 0.05,
    "review_corr_min": 0.975,
    "review_gradient_min": 0.93,
    "review_aspect_max": 0.1,
    "max_private_contact_pairs": 12,
    "note": "All 0..6 pHash candidates will be compared; HIGH/REVIEW are not confirmed duplicates."
  },
  "release": {
    "automatic_patient_independence_claim": false,
    "automatic_model_training_approval": false,
    "research_use_only": true,
    "requires_human_review_and_documented_risk_acceptance": true
  },
  "resume_from_previous_phase4": {
    "allowed": true,
    "required_prior_policy_version": "pneumolens-split-v1",
    "required_prior_inventory_verified": true,
    "if_unavailable": "full audit from images, with progress reporting",
    "identity_check": "previous full inventory file SHA matched Phase 1; current paths match; candidate source bytes are reverified before scoring"
  }
}
'''
POLICY = json.loads(POLICY_TEXT)
(PROJECT_ROOT / 'configs/split_policy.json').write_text(POLICY_TEXT)
print('Policy:', POLICY['version'])
print('Expected source SHA-256 inventory:', POLICY['source']['expected_dataset_file_sha256_fingerprint'])

Policy: pneumolens-split-v2
Expected source SHA-256 inventory: a7b36506fbccf5f77b5d80a73b7a53cca8515dba45ae959a8d0a62e5f219a87b


### 3. Reusable module — single source of truth
This exported source provides dataset discovery, strict identity checks, all-pairs pHash screening, image-level similarity scoring, grouping, deterministic validation splitting, and the decision gate. Keep the resulting `src/data/split.py` for Phase 5.

In [3]:
MODULE_SOURCE = r'''

"""PneumoLens Phase 4 V2: complete, bounded near-duplicate screening.

This module does NOT establish patient independence without trusted provenance.
It only constructs a candidate dev split while keeping the original test locked.
For research/education, not clinical use.
"""
from __future__ import annotations

import hashlib
import io
import os
import re
from collections import defaultdict
from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold

LABELS = ("NORMAL", "PNEUMONIA")
CLASS_TO_INDEX = {"NORMAL": 0, "PNEUMONIA": 1}
SUPPORTED_EXTENSIONS = {".jpeg", ".jpg", ".png"}
FILENAME_RE = re.compile(r"^(NORMAL|BACTERIA|VIRUS)-(\d+)-(\d+)\.(jpe?g|png)$", re.I)


@dataclass
class Source:
    kind: str
    location: Path
    zip_prefix: str = ""
    _archive: ZipFile | None = field(default=None, repr=False, compare=False)

    def iter_images(self):
        if self.kind == "folder":
            for original_split in ("train", "test"):
                for label in LABELS:
                    for p in sorted((self.location / original_split / label).iterdir()):
                        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS:
                            yield original_split, label, p.relative_to(self.location).as_posix()
        elif self.kind == "zip":
            with ZipFile(self.location) as archive:
                members = sorted(n for n in archive.namelist() if not n.endswith("/"))
            for member in members:
                if not member.startswith(self.zip_prefix):
                    continue
                rel = member[len(self.zip_prefix):]
                parts = rel.split("/")
                if (len(parts) == 3 and parts[0].lower() in ("train", "test")
                        and parts[1].upper() in LABELS and Path(parts[2]).suffix.lower() in SUPPORTED_EXTENSIONS):
                    yield parts[0].lower(), parts[1].upper(), "/".join(parts)
        else:
            raise ValueError(f"Unsupported source kind: {self.kind}")

    def read(self, relative_path: str) -> bytes:
        if self.kind == "folder":
            p = (self.location / relative_path).resolve()
            if not p.is_relative_to(self.location.resolve()):
                raise ValueError("Unsafe relative path outside data source")
            return p.read_bytes()
        if self._archive is None:
            self._archive = ZipFile(self.location)
        return self._archive.read(self.zip_prefix + relative_path)


def _valid_root(path: Path) -> bool:
    return all((path / sp / lab).is_dir() for sp in ("train", "test") for lab in LABELS)


def _zip_prefixes(archive_path: Path) -> list[str]:
    try:
        with ZipFile(archive_path) as archive:
            names = [n.replace("\\", "/") for n in archive.namelist()]
    except Exception:
        return []
    patterns = defaultdict(set)
    for name in names:
        if Path(name).suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue
        m = re.match(r"^(.*?/)?(train|test)/(NORMAL|PNEUMONIA)/[^/]+$", name, re.I)
        if m:
            patterns[m.group(1) or ""].add((m.group(2).lower(), m.group(3).upper()))
    required = {(sp, lab) for sp in ("train", "test") for lab in LABELS}
    return sorted(p for p, present in patterns.items() if required.issubset(present))


def discover_source(explicit: str | None = None, search_root: str = "/kaggle/input") -> Source:
    """Reject ambiguous or missing datasets. No silent OCT fallback."""
    preferred = Path("/kaggle/input/datasets/srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray")
    override = explicit or os.environ.get("PNEUMOLENS_DATASET_ROOT", "")
    if override:
        path = Path(override)
        if path.is_dir() and _valid_root(path):
            return Source("folder", path)
        if path.is_file() and path.suffix.lower() == ".zip":
            options = _zip_prefixes(path)
            if len(options) == 1:
                return Source("zip", path, options[0])
        raise FileNotFoundError(f"Explicit source is not a unique chest_xray train/test dataset: {override}")
    if _valid_root(preferred):
        return Source("folder", preferred)
    base = Path(search_root)
    if not base.exists():
        raise FileNotFoundError(f"Dataset root unavailable: {base}")
    folders = []
    archives = []
    for dirpath, dirs, filenames in os.walk(base):
        current = Path(dirpath)
        if _valid_root(current):
            folders.append(current)
            dirs[:] = []
            continue
        # Avoid traversing the 100k+ OCT files in mixed ZhangLab archives.
        dirs[:] = [x for x in dirs if x.lower() not in {"oct", "optical_coherence_tomography", "__macosx", ".git"}]
        archives.extend(current / fn for fn in filenames if fn.lower().endswith(".zip"))
    if len(folders) == 1:
        return Source("folder", folders[0])
    if len(folders) > 1:
        raise RuntimeError(f"Multiple chest_xray sources found. Set PNEUMOLENS_DATASET_ROOT: {folders[:4]}")
    candidates = [(f, prefix) for f in archives for prefix in _zip_prefixes(f)]
    if len(candidates) == 1:
        return Source("zip", candidates[0][0], candidates[0][1])
    raise RuntimeError(f"Expected one chest X-ray folder or ZIP, found {len(candidates)} options")


def series_candidate(filename: str) -> tuple[str, bool]:
    """Weak same-series key only. Prefix+number is NOT a patient identifier."""
    m = FILENAME_RE.match(filename)
    if m:
        return f"filename-series:{m.group(1).upper()}-{m.group(2)}", True
    return f"unparsed-filename:{filename}", False


def dct_basis(size: int = 32) -> np.ndarray:
    a = np.arange(size, dtype=np.float64)
    k = a.reshape(-1, 1)
    basis = np.cos(np.pi * (2 * a + 1) * k / (2 * size))
    basis[0, :] *= np.sqrt(1 / size)
    basis[1:, :] *= np.sqrt(2 / size)
    return basis


_DCT32 = dct_basis(32)


def perceptual_hash(gray_thumbnail: np.ndarray) -> int:
    """64-bit pHash (DCT), independent of Phase 1's collision-prone dHash."""
    arr = np.asarray(Image.fromarray(gray_thumbnail, "L").resize((32, 32), Image.Resampling.BILINEAR), dtype=np.float64)
    coeff = _DCT32 @ arr @ _DCT32.T
    block = coeff[:8, :8].copy()
    median = np.median(block.flatten()[1:])
    bits = (block > median).flatten()
    bits[0] = False
    return sum(1 << i for i, bit in enumerate(bits) if bit)


def analyze_source(source: Source, fail_fast: bool = True) -> tuple[pd.DataFrame, np.ndarray]:
    """Single read/decode per image, secure hashes and compact private image signatures."""
    rows: list[dict] = []
    thumbs_by_path = {}
    try:
        from tqdm.auto import tqdm
        images = tqdm(list(source.iter_images()), desc="Building source inventory (first-run fallback)", unit="image")
    except ImportError:
        images = source.iter_images()
    for original_split, label, rel in images:
        filename = Path(rel).name
        series_key, parsed = series_candidate(filename)
        try:
            raw = source.read(rel)
            file_sha = hashlib.sha256(raw).hexdigest()
            with Image.open(io.BytesIO(raw)) as image:
                transposed = ImageOps.exif_transpose(image)
                gray = transposed.convert("L")
                width, height = gray.size
                pixel_sha = hashlib.sha256(f"{width}x{height}:".encode("ascii") + gray.tobytes()).hexdigest()
                thumb = np.asarray(gray.resize((96, 96), Image.Resampling.BILINEAR), dtype=np.uint8)
                image_mode = image.mode
            rows.append({
                "relative_path": rel, "filename": filename, "original_split": original_split,
                "label": label, "label_index": CLASS_TO_INDEX[label],
                "file_sha256": file_sha, "decoded_sha256": pixel_sha,
                "phash64": f"{perceptual_hash(thumb):016x}",
                "series_group_UNVERIFIED": series_key, "filename_series_parse_ok": parsed,
                "width": width, "height": height, "aspect_ratio": width / height,
                "image_mode": image_mode, "bytes": len(raw), "decode_ok": True,
            })
            thumbs_by_path[rel] = thumb
        except Exception as exc:
            if fail_fast:
                raise RuntimeError(f"Unable to inspect {rel}: {type(exc).__name__}: {exc}") from exc
            rows.append({"relative_path": rel, "filename": filename, "original_split": original_split,
                         "label": label, "decode_ok": False, "error": str(exc)})
    manifest = pd.DataFrame(rows).sort_values("relative_path").reset_index(drop=True)
    if manifest.empty:
        raise RuntimeError("No supported chest X-rays found")
    if not manifest.decode_ok.all():
        raise RuntimeError("Integrity failed: unreadable images detected")
    # Thumbnails must use exactly the same row order as the sorted manifest.
    if not manifest["relative_path"].is_unique:
        raise RuntimeError("Duplicate relative paths detected")
    thumb_stack = np.stack([thumbs_by_path[rel] for rel in manifest.relative_path])
    if source._archive is not None:
        source._archive.close()
        source._archive = None
    return manifest, thumb_stack


def duplicate_violations(df: pd.DataFrame) -> dict:
    out = {}
    for key in ("file_sha256", "decoded_sha256"):
        grouped = df.groupby(key, dropna=False)
        out[f"{key}_cross_original_split_groups"] = int((grouped["original_split"].nunique() > 1).sum())
        out[f"{key}_conflicting_label_groups"] = int((grouped["label"].nunique() > 1).sum())
        out[f"{key}_duplicate_groups"] = int((grouped.size() > 1).sum())
    return out


class DisjointSet:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, i):
        while self.parent[i] != i:
            self.parent[i] = self.parent[self.parent[i]]
            i = self.parent[i]
        return i

    def merge(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return
        if self.rank[a] < self.rank[b]:
            a, b = b, a
        self.parent[b] = a
        if self.rank[a] == self.rank[b]:
            self.rank[a] += 1



def exhaustive_phash_candidates(df: pd.DataFrame, max_distance: int = 6,
                                block_size: int = 128) -> tuple[pd.DataFrame, dict]:
    """Complete 64-bit pHash radius search: O(n^2) time, O(block*n) memory.

    Every unordered pair is evaluated; there is NO band-hash budget, sampling,
    early termination or implicit guarantee that pHash discovers all duplicates.
    """
    if not (0 <= max_distance <= 64):
        raise ValueError("max_distance must be in [0, 64]")
    if not 1 <= block_size <= 1024:
        raise ValueError("block_size must be in [1, 1024]")
    values = np.asarray([int(str(x), 16) for x in df.phash64], dtype=np.uint64)
    n = len(values)
    lookup = np.asarray([int(i).bit_count() for i in range(256)], dtype=np.uint8)
    records = []
    for start in range(0, n, block_size):
        stop = min(n, start + block_size)
        # Only comparisons where the other index > start are materialized.
        xors = np.bitwise_xor(values[start:stop, None], values[None, start + 1:])
        byte_view = np.ascontiguousarray(xors).view(np.uint8).reshape(stop - start, n - start - 1, 8)
        distance = np.zeros(xors.shape, dtype=np.uint8)
        for offset in range(8):
            distance += lookup[byte_view[:, :, offset]]
        a, b = np.where((distance <= max_distance) &
                        (np.arange(start, stop)[:, None] < np.arange(start + 1, n)[None, :]))
        records.extend((int(start + i), int(start + 1 + j), int(distance[i, j]))
                       for i, j in zip(a, b))
    result = pd.DataFrame(records, columns=["index_a", "index_b", "phash_hamming"])
    if not result.empty:
        result = result.sort_values(["phash_hamming", "index_a", "index_b"]).reset_index(drop=True)
    diagnostics = {
        "total_image_pairs_compared": n * (n - 1) // 2,
        "phash_candidates": len(result),
        "phash_distance_threshold": int(max_distance),
        "complete_exhaustive_phash_search": True,
        "screen_incomplete_due_to_resource_budget": False,
        "method": "All-pairs vectorized 64-bit pHash Hamming distance; no LSH cap",
        "limitation": "pHash radius does not guarantee detection of every visually similar radiograph or shared patient",
    }
    return result, diagnostics


def load_candidate_thumbnails(source: Source, df: pd.DataFrame,
                              candidates: pd.DataFrame,
                              full_thumbnails: np.ndarray | None = None) -> dict[int, np.ndarray]:
    """Read each candidate image ONCE, verifying original bytes against cached inventory."""
    if candidates.empty:
        return {}
    indices = sorted(set(candidates.index_a.astype(int)) | set(candidates.index_b.astype(int)))
    if full_thumbnails is not None:
        if len(full_thumbnails) != len(df):
            raise ValueError("Full thumbnail array is not aligned with the cached inventory")
        return {idx: full_thumbnails[idx] for idx in indices}
    try:
        from tqdm.auto import tqdm
        iterator = tqdm(indices, desc="Checking only pHash candidate X-rays", unit="image")
    except ImportError:
        iterator = indices
    cache: dict[int, np.ndarray] = {}
    for idx in iterator:
        row = df.iloc[idx]
        raw = source.read(str(row.relative_path))
        sha = hashlib.sha256(raw).hexdigest()
        if sha != row.file_sha256:
            raise RuntimeError(f"Source bytes changed since Phase 4 v1 for {row.relative_path}")
        with Image.open(io.BytesIO(raw)) as image:
            gray = ImageOps.exif_transpose(image).convert("L")
            thumb = np.asarray(gray.resize((96, 96), Image.Resampling.BILINEAR), dtype=np.uint8)
        if f"{perceptual_hash(thumb):016x}" != row.phash64:
            raise RuntimeError(f"pHash changed since Phase 4 v1 for {row.relative_path}")
        cache[idx] = thumb
    if source._archive is not None:
        source._archive.close()
        source._archive = None
    return cache


def score_phash_candidates(df: pd.DataFrame, candidates: pd.DataFrame,
                           thumbs: dict[int, np.ndarray], policy: dict) -> pd.DataFrame:
    """Record EVERY pHash candidate and its measured pixel-structure evidence.

    HIGH / REVIEW are heuristic manual-review flags, never confirmed duplicates.
    LOW_SCREENED means the chosen image-structure cutoffs were not met.
    """
    p = policy["near_duplicate_screen"]
    records = []
    for a, b, hd in candidates[["index_a", "index_b", "phash_hamming"]].itertuples(index=False, name=None):
        ra, rb = df.iloc[int(a)], df.iloc[int(b)]
        exact = (ra.file_sha256 == rb.file_sha256 or ra.decoded_sha256 == rb.decoded_sha256)
        aspect = abs(float(ra.aspect_ratio) - float(rb.aspect_ratio)) / max(float(ra.aspect_ratio), float(rb.aspect_ratio))
        corr, grad, mae = pair_similarity(thumbs[int(a)], thumbs[int(b)])
        high = (not exact and hd <= int(p["strong_phash_max"]) and
                corr >= float(p["strong_corr_min"]) and
                grad >= float(p["strong_gradient_min"]) and
                mae <= float(p["strong_mae_max"]) and
                aspect <= float(p["strong_aspect_max"]))
        review = (not exact and corr >= float(p["review_corr_min"]) and
                  grad >= float(p["review_gradient_min"]) and
                  aspect <= float(p["review_aspect_max"]))
        priority = ("EXACT_KNOWN" if exact else "HIGH" if high else "REVIEW" if review else "LOW_SCREENED")
        records.append({
            "index_a": int(a), "index_b": int(b), "path_a": str(ra.relative_path), "path_b": str(rb.relative_path),
            "label_a": ra.label, "label_b": rb.label, "source_split_a": ra.original_split,
            "source_split_b": rb.original_split, "phash_hamming": int(hd),
            "pixel_corr": round(float(corr), 7), "gradient_corr": round(float(grad), 7),
            "pixel_mae_fraction": round(float(mae), 7), "aspect_relative_delta": round(float(aspect), 7),
            "screening_priority": priority, "confirmed_duplicate": bool(exact),
            "human_decision": "EXACT_ALREADY_HANDLED" if exact else "PENDING_REVIEW" if priority in ("HIGH", "REVIEW") else "NOT_FLAGGED_BY_SCREEN",
        })
    cols = ["index_a", "index_b", "path_a", "path_b", "label_a", "label_b",
            "source_split_a", "source_split_b", "phash_hamming", "pixel_corr",
            "gradient_corr", "pixel_mae_fraction", "aspect_relative_delta",
            "screening_priority", "confirmed_duplicate", "human_decision"]
    frame = pd.DataFrame.from_records(records, columns=cols)
    if not frame.empty:
        order = {"HIGH": 0, "REVIEW": 1, "LOW_SCREENED": 2, "EXACT_KNOWN": 3}
        frame["_priority_rank"] = frame.screening_priority.map(order)
        frame = frame.sort_values(["_priority_rank", "phash_hamming", "pixel_corr", "index_a", "index_b"],
                                  ascending=[True, True, False, True, True]).drop(columns=["_priority_rank"]).reset_index(drop=True)
    return frame


def action_review_queue(scored: pd.DataFrame) -> pd.DataFrame:
    return scored[scored.screening_priority.isin(["HIGH", "REVIEW"])].copy().reset_index(drop=True)


def screen_image_similarity(df: pd.DataFrame, thumbs: np.ndarray, policy: dict):
    """Backwards-compatible helper for unit tests with resident thumbnails."""
    candidates, diagnostics = exhaustive_phash_candidates(df, int(policy["near_duplicate_screen"]["max_phash_distance"]))
    cache = load_candidate_thumbnails(None, df, candidates, full_thumbnails=thumbs) if len(candidates) else {}
    full = score_phash_candidates(df, candidates, cache, policy)
    queue = action_review_queue(full)
    diagnostics.update({"phash_screened_pairs": len(full), "review_queue_pairs": len(queue),
                        "high_priority_pairs": int((queue.screening_priority == "HIGH").sum()),
                        "all_phash_candidates_image_scored": len(full) == len(candidates)})
    return queue, diagnostics

def _normalized_corr(a: np.ndarray, b: np.ndarray) -> float:
    x = a.astype(np.float32).ravel()
    y = b.astype(np.float32).ravel()
    x -= x.mean(); y -= y.mean()
    denominator = float(np.linalg.norm(x) * np.linalg.norm(y))
    if denominator < 1e-9:
        return 1.0 if np.array_equal(a, b) else 0.0
    return float(np.clip(np.dot(x, y) / denominator, -1.0, 1.0))


def pair_similarity(a: np.ndarray, b: np.ndarray) -> tuple[float, float, float]:
    corr = _normalized_corr(a, b)
    dx_a = np.diff(a.astype(np.float32), axis=1)
    dx_b = np.diff(b.astype(np.float32), axis=1)
    dy_a = np.diff(a.astype(np.float32), axis=0)
    dy_b = np.diff(b.astype(np.float32), axis=0)
    grad_corr = (_normalized_corr(dx_a, dx_b) + _normalized_corr(dy_a, dy_b)) / 2
    mae = float(np.abs(a.astype(np.float32) - b.astype(np.float32)).mean() / 255.0)
    return corr, grad_corr, mae


def _join_duplicates(ds: DisjointSet, df: pd.DataFrame, keys: tuple[str, ...]):
    for key in keys:
        for _, group in df.groupby(key, sort=True):
            members = group.index.to_list()
            for j in members[1:]:
                ds.merge(members[0], j)


def build_split(df: pd.DataFrame, pairs: pd.DataFrame, policy: dict) -> tuple[pd.DataFrame, dict]:
    """Never uses original test rows in selecting candidate validation fold."""
    df = df.copy().reset_index(drop=True)
    # Stable order required for hashes, indices and reproducibility.
    if not df.relative_path.is_monotonic_increasing:
        raise ValueError("Manifest must be sorted by relative_path before splitting")
    train = df.index[df.original_split.eq("train")].to_list()
    test = df.index[df.original_split.eq("test")].to_list()
    ds = DisjointSet(len(df))
    _join_duplicates(ds, df, ("file_sha256", "decoded_sha256", "series_group_UNVERIFIED"))
    if not pairs.empty:
        for row in pairs.itertuples(index=False):
            if row.screening_priority == "HIGH" and row.source_split_a == row.source_split_b == "train":
                ds.merge(int(row.index_a), int(row.index_b))
    df["candidate_group"] = [f"G{ds.find(i):06d}" for i in range(len(df))]
    # Deduplicate only original train; original test is never reduced or reallocated.
    df["split_role"] = "TEST_LOCKED"
    df["duplicate_of"] = ""
    df.loc[train, "split_role"] = "UNASSIGNED_TRAIN_SOURCE"
    for _, grp in df.loc[train].groupby("decoded_sha256", sort=True):
        members = list(grp.index)
        keeper = min(members, key=lambda i: df.at[i, "relative_path"])
        for idx in members:
            if idx != keeper:
                df.at[idx, "split_role"] = "EXCLUDED_EXACT_COPY"
                df.at[idx, "duplicate_of"] = df.at[keeper, "relative_path"]
    # Cross-encoded file-byte exact copies may also have the same pixel hash;
    # decoded_sha256 is the deduplication criterion.
    retained = df.loc[df.split_role.eq("UNASSIGNED_TRAIN_SOURCE")]
    groups = retained.candidate_group.to_numpy()
    y = retained.label_index.to_numpy(dtype=np.int64)
    if len(set(groups)) < 5 or set(y) != {0, 1}:
        raise RuntimeError("Too few groups or missing classes for 5-fold split")
    n_splits = int(policy["validation"]["n_splits"])
    choices = []
    for seed in policy["validation"]["candidate_seeds"]:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=int(seed))
        for fold, (fit_ix, val_ix) in enumerate(splitter.split(np.zeros(len(y)), y, groups)):
            yp = y[val_ix]
            if len(set(yp)) != 2 or len(set(y[fit_ix])) != 2:
                continue
            fraction = len(val_ix) / len(y)
            val_p = yp.mean()
            global_p = y.mean()
            fit_p = y[fit_ix].mean()
            score = 3 * abs(fraction - 1 / n_splits) + 2 * abs(val_p - global_p) + abs(fit_p - global_p)
            choices.append((score, int(seed), int(fold), fit_ix, val_ix))
    if not choices:
        raise RuntimeError("No grouped split retained both classes")
    best = min(choices, key=lambda c: (round(c[0], 12), c[1], c[2]))
    retained_idxs = retained.index.to_numpy()
    df.loc[retained_idxs[best[3]], "split_role"] = "TRAIN"
    df.loc[retained_idxs[best[4]], "split_role"] = "VALIDATION"
    assigned = df.split_role.value_counts().to_dict()
    train_groups = set(df.loc[df.split_role.eq("TRAIN"), "candidate_group"])
    val_groups = set(df.loc[df.split_role.eq("VALIDATION"), "candidate_group"])
    test_groups = set(df.loc[df.split_role.eq("TEST_LOCKED"), "candidate_group"])
    train_hashes = set(df.loc[df.split_role.eq("TRAIN"), "decoded_sha256"])
    val_hashes = set(df.loc[df.split_role.eq("VALIDATION"), "decoded_sha256"])
    test_hashes = set(df.loc[df.split_role.eq("TEST_LOCKED"), "decoded_sha256"])
    original_test = set(df.loc[df.original_split.eq("test"), "relative_path"])
    output_test = set(df.loc[df.split_role.eq("TEST_LOCKED"), "relative_path"])
    violations = duplicate_violations(df)
    cross_source_series = int((df.groupby("series_group_UNVERIFIED").original_split.nunique() > 1).sum())
    qa = {
        "all_rows_accounted_for": int(sum(assigned.values())) == len(df),
        "original_test_exactly_preserved": original_test == output_test,
        "train_validation_groups_disjoint": train_groups.isdisjoint(val_groups),
        "train_test_heuristic_groups_disjoint": train_groups.isdisjoint(test_groups),
        "validation_test_heuristic_groups_disjoint": val_groups.isdisjoint(test_groups),
        "train_validation_pixel_sha_disjoint": train_hashes.isdisjoint(val_hashes),
        "train_test_pixel_sha_disjoint": train_hashes.isdisjoint(test_hashes),
        "validation_test_pixel_sha_disjoint": val_hashes.isdisjoint(test_hashes),
        "no_pixel_sha_label_conflicts": violations["decoded_sha256_conflicting_label_groups"] == 0,
        "no_file_sha_label_conflicts": violations["file_sha256_conflicting_label_groups"] == 0,
        "no_heuristic_series_cross_original_test": cross_source_series == 0,
        "all_original_train_accounted_for": int((df.original_split == "train").sum()) == int((df.split_role != "TEST_LOCKED").sum()),
        "both_fit_classes_present": set(df.loc[df.split_role.eq("TRAIN"), "label"]) == set(LABELS),
        "both_validation_classes_present": set(df.loc[df.split_role.eq("VALIDATION"), "label"]) == set(LABELS),
    }
    retained_val = df[df.split_role == "VALIDATION"]
    retained_fit = df[df.split_role == "TRAIN"]
    source_p = float(retained.label_index.mean())
    val_p = float(retained_val.label_index.mean())
    val_fraction = len(retained_val) / len(retained)
    qa["validation_fraction_within_declared_tolerance"] = (
        abs(val_fraction - float(policy["validation"]["target_fraction"])) <=
        float(policy["validation"]["max_abs_fraction_deviation"]))
    qa["validation_class_rate_within_declared_tolerance"] = (
        abs(val_p - source_p) <=
        float(policy["validation"]["max_abs_positive_rate_deviation"]))
    qa["no_high_priority_cross_original_source"] = True if pairs.empty else not any(
        (pairs.screening_priority == "HIGH") & (pairs.source_split_a != pairs.source_split_b))
    qa["no_high_priority_cross_train_val"] = True
    qa["pending_cross_partition_review_pairs"] = 0
    if not pairs.empty:
        role_map = df.split_role.to_dict()
        split_roles = []
        for row in pairs.itertuples(index=False):
            a, b = role_map[int(row.index_a)], role_map[int(row.index_b)]
            roles = {a, b}
            crosses = (a != b and not roles.issubset({"TRAIN", "EXCLUDED_EXACT_COPY"})
                       and not roles.issubset({"VALIDATION", "EXCLUDED_EXACT_COPY"}))
            split_roles.append((a, b, crosses))
        pairs["role_a"] = [r[0] for r in split_roles]
        pairs["role_b"] = [r[1] for r in split_roles]
        pairs["cross_candidate_partitions"] = [r[2] for r in split_roles]
        qa["no_high_priority_cross_train_val"] = not any(
            (pairs.screening_priority == "HIGH") &
            (pairs.role_a.isin(["TRAIN", "VALIDATION"])) &
            (pairs.role_b.isin(["TRAIN", "VALIDATION"])) &
            (pairs.role_a != pairs.role_b))
        qa["pending_cross_partition_review_pairs"] = int(pairs.cross_candidate_partitions.sum())
    assigned = df.split_role.value_counts().to_dict()
    locked_test_exact_group_sizes = (df[df.original_split.eq("test")]
                                     .groupby("decoded_sha256").size())
    locked_test_redundant_exact_copies = int((locked_test_exact_group_sizes - 1).clip(lower=0).sum())
    counts = df[df.split_role.isin(["TRAIN", "VALIDATION"])].groupby(["split_role", "label"]).size().to_dict()
    c0, c1 = int(counts.get(("TRAIN", "NORMAL"), 0)), int(counts.get(("TRAIN", "PNEUMONIA"), 0))
    weights = {"NORMAL": (c0 + c1) / (2 * c0), "PNEUMONIA": (c0 + c1) / (2 * c1)}
    diagnostics = {
        "selected_seed": best[1], "selected_fold": best[2],
        "selected_score": float(best[0]), "tested_splits": len(choices),
        "source_count": len(df), "source_train_count": len(train), "source_test_count": len(test),
        "dedup_removed_from_original_train": int(assigned.get("EXCLUDED_EXACT_COPY", 0)),
        "locked_test_redundant_exact_copies_preserved": locked_test_redundant_exact_copies,
        "role_counts": {str(k): int(v) for k, v in assigned.items()},
        "validation_fraction_of_retained_train": float(assigned.get("VALIDATION", 0) / (assigned.get("VALIDATION", 0) + assigned.get("TRAIN", 0))),
        "class_counts": {f"{role}/{label}": int(v) for (role, label), v in counts.items()},
        "class_weights_train_only": weights, "qa": qa, "duplicate_diagnostics": violations,
        "series_groups_cross_original_split": cross_source_series,
        "verified_patient_id_count": 0,
        "patient_independence_verified": False,
    }
    return df, diagnostics


def split_fingerprint(df: pd.DataFrame) -> str:
    canonical = "\n".join(f"{r.relative_path}|{r.split_role}|{r.decoded_sha256}" for r in
                           df.sort_values("relative_path").itertuples(index=False))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def training_gate(diag: dict, screen: dict, policy: dict) -> dict:
    qa = diag["qa"]
    boolean_checks = {k: v for k, v in qa.items() if isinstance(v, bool)}
    technical_pass = (all(boolean_checks.values()) and
                      screen.get("complete_exhaustive_phash_search", False) and
                      screen.get("all_phash_candidates_image_scored", False) and
                      not screen.get("screen_incomplete_due_to_resource_budget", True))
    # 'Technical pass' alone never grants research/clinical training approval.
    return {
        "technical_split_gate": "PASS_AUTOMATED_CHECKS_WITH_LIMITATIONS" if technical_pass else "FAIL_NEEDS_INVESTIGATION",
        "research_training_gate": "HOLD_PENDING_HUMAN_REVIEW_AND_RISK_ACCEPTANCE",
        "patient_independence": "UNVERIFIED",
        "clinical_validation": "NOT_SUPPORTED",
        "why": [
            "Filename series keys are not trusted patient identifiers",
            "Any HIGH/REVIEW candidates need human adjudication; a negative pHash screen does not establish patient independence",
            "Original test is locked for model selection and threshold tuning",
        ],
        "preregistered_thresholds": policy["near_duplicate_screen"],
    }


def inventory_fingerprint(df: pd.DataFrame) -> str:
    canonical = "\n".join(f"{r.relative_path}|{r.file_sha256}" for r in
                          df.sort_values("relative_path").itertuples(index=False))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

'''
MODULE_PATH = PROJECT_ROOT / 'src/data/split.py'
MODULE_PATH.write_text(MODULE_SOURCE)
from src.data.split import (discover_source, analyze_source, inventory_fingerprint,
    exhaustive_phash_candidates, load_candidate_thumbnails, score_phash_candidates,
    action_review_queue, build_split, training_gate, split_fingerprint, duplicate_violations)
print('Exported module:', MODULE_PATH, 'SHA256:', hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest())

Exported module: /kaggle/working/pneumolens_phase4_v2/project_files/src/data/split.py SHA256: 29123b972027fa98f0b92ec9d701635b94d68aa1782fd65d2d76ea122b9d18fc


### 4. Run the unit tests before inspecting the dataset
Tests cover split reproducibility and locked-test protection as well as exact completeness of the new pHash search against an independent brute-force implementation.

In [4]:
TEST_SOURCE = r'''

"""Phase 4 unit tests: run with `python -m unittest discover -s tests`."""
import sys
import unittest
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import numpy as np
import pandas as pd
from src.data.split import (
    DisjointSet, build_split, duplicate_violations, pair_similarity,
    perceptual_hash, series_candidate, split_fingerprint, training_gate,
    screen_image_similarity,
)


class TestSplitIntegrity(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.policy = {
            "validation": {"n_splits": 5, "candidate_seeds": [2026, 2027, 2031],
                           "target_fraction": 0.20, "max_abs_fraction_deviation": 0.10,
                           "max_abs_positive_rate_deviation": 0.10},
            "near_duplicate_screen": {"max_phash_distance": 6},
        }

    def fixture(self, conflict=False, cross_test_exact=False):
        rows = []
        for source_split in ("train", "test"):
            for label in ("NORMAL", "PNEUMONIA"):
                count = 40 if source_split == "train" else 10
                for k in range(count):
                    token = 10000 + k
                    prefix = "NORMAL" if label == "NORMAL" else "BACTERIA"
                    file = f"{prefix}-{token}-0001.jpeg"
                    rel = f"{source_split}/{label}/{file}"
                    signature = f"HASH:{source_split}:{label}:{k}"
                    rows.append({"relative_path": rel, "original_split": source_split,
                                 "label": label, "label_index": 0 if label == "NORMAL" else 1,
                                 "series_group_UNVERIFIED": series_candidate(file)[0],
                                 "file_sha256": signature, "decoded_sha256": signature,
                                 "phash64": f"{k:016x}", "filename": file})
        out = pd.DataFrame(rows).sort_values("relative_path").reset_index(drop=True)
        # A repeat under a different numeric token should not cross validation.
        train_normal = out.index[(out.original_split == "train") & (out.label == "NORMAL")]
        out.loc[train_normal[1], ["file_sha256", "decoded_sha256"]] = out.loc[train_normal[0], ["file_sha256", "decoded_sha256"]].to_numpy()
        if cross_test_exact:
            test_normal = out.index[(out.original_split == "test") & (out.label == "NORMAL")]
            out.loc[test_normal[0], ["file_sha256", "decoded_sha256"]] = out.loc[train_normal[0], ["file_sha256", "decoded_sha256"]].to_numpy()
        if conflict:
            train_p = out.index[(out.original_split == "train") & (out.label == "PNEUMONIA")]
            out.loc[train_p[0], ["file_sha256", "decoded_sha256"]] = out.loc[train_normal[0], ["file_sha256", "decoded_sha256"]].to_numpy()
        return out

    def test_series_key_preserves_prefix(self):
        self.assertNotEqual(series_candidate("BACTERIA-1234-0001.jpeg")[0],
                            series_candidate("VIRUS-1234-0001.jpeg")[0])

    def test_unparsed_token_not_patient(self):
        self.assertFalse(series_candidate("person3.jpeg")[1])

    def test_dct_phash_repeatable(self):
        rng = np.random.default_rng(0)
        image = rng.integers(0, 255, (96, 96), dtype=np.uint8)
        self.assertEqual(perceptual_hash(image), perceptual_hash(image.copy()))
        self.assertTrue(isinstance(perceptual_hash(image), int))

    def test_image_similarity_identity(self):
        image = np.arange(96 * 96, dtype=np.uint32).reshape(96, 96).astype(np.uint8)
        corr, grad_corr, mae = pair_similarity(image, image.copy())
        self.assertAlmostEqual(corr, 1, places=5)
        self.assertAlmostEqual(grad_corr, 1, places=5)
        self.assertEqual(mae, 0)

    def test_disjoint_set(self):
        ds = DisjointSet(5)
        ds.merge(0, 3)
        self.assertEqual(ds.find(0), ds.find(3))
        self.assertNotEqual(ds.find(1), ds.find(3))

    def test_train_test_duplicate_detected(self):
        v = duplicate_violations(self.fixture(cross_test_exact=True))
        self.assertGreater(v["decoded_sha256_cross_original_split_groups"], 0)

    def test_conflicting_label_detected(self):
        v = duplicate_violations(self.fixture(conflict=True))
        self.assertGreater(v["decoded_sha256_conflicting_label_groups"], 0)

    def test_candidate_split_reproducible(self):
        d = self.fixture()
        pair = pd.DataFrame()
        a, diag_a = build_split(d, pair, self.policy)
        b, diag_b = build_split(d, pair.copy(), self.policy)
        self.assertEqual(split_fingerprint(a), split_fingerprint(b))
        self.assertEqual(diag_a["selected_fold"], diag_b["selected_fold"])

    def test_locked_test_and_dedup(self):
        d, stats = build_split(self.fixture(), pd.DataFrame(), self.policy)
        self.assertTrue(stats["qa"]["original_test_exactly_preserved"])
        self.assertEqual(int((d.split_role == "TEST_LOCKED").sum()), 20)
        self.assertEqual(int((d.split_role == "EXCLUDED_EXACT_COPY").sum()), 1)
        self.assertTrue(stats["qa"]["train_validation_groups_disjoint"])

    def test_strong_grouping_does_not_cross_fit_val(self):
        raw = self.fixture()
        locs = raw.index[(raw.original_split == "train") & (raw.label == "NORMAL")]
        pa, pb = int(locs[3]), int(locs[4])
        paired = pd.DataFrame([{
            "index_a": pa, "index_b": pb,
            "source_split_a": "train", "source_split_b": "train",
            "screening_priority": "HIGH"}])
        d, diag = build_split(raw, paired, self.policy)
        self.assertEqual(d.at[pa, "candidate_group"], d.at[pb, "candidate_group"])
        self.assertTrue(diag["qa"]["no_high_priority_cross_train_val"])

    def test_training_gate_never_automatically_approves(self):
        _, stats = build_split(self.fixture(), pd.DataFrame(), self.policy)
        result = training_gate(stats, {"screen_incomplete_due_to_resource_budget": False,
                                      "complete_exhaustive_phash_search": True,
                                      "all_phash_candidates_image_scored": True}, self.policy)
        self.assertEqual(result["research_training_gate"], "HOLD_PENDING_HUMAN_REVIEW_AND_RISK_ACCEPTANCE")
        self.assertEqual(result["patient_independence"], "UNVERIFIED")

    def test_near_match_screen_flags_structurally_identical_pair(self):
        rng = np.random.default_rng(444)
        image = rng.integers(20, 210, (96, 96), dtype=np.uint8)
        transformed = np.clip(image.astype(np.int16) + 1, 0, 255).astype(np.uint8)
        df = pd.DataFrame([
            {"phash64": f"{perceptual_hash(image):016x}", "relative_path": "train/NORMAL/one.jpeg",
             "file_sha256": "f1", "decoded_sha256": "p1", "original_split": "train", "label": "NORMAL",
             "aspect_ratio": 1.0},
            {"phash64": f"{perceptual_hash(transformed):016x}", "relative_path": "test/NORMAL/two.jpeg",
             "file_sha256": "f2", "decoded_sha256": "p2", "original_split": "test", "label": "NORMAL",
             "aspect_ratio": 1.0},
        ])
        limits = {"near_duplicate_screen": {
            "max_phash_distance": 6, "max_band_candidate_pairs": 1000, "max_scored_pairs": 1000,
            "strong_phash_max": 3, "strong_corr_min": 0.995, "strong_gradient_min": 0.975,
            "strong_mae_max": 0.045, "strong_aspect_max": 0.05,
            "review_corr_min": 0.975, "review_gradient_min": 0.93, "review_aspect_max": 0.10,
        }}
        queue, diagnostic = screen_image_similarity(df, np.stack([image, transformed]), limits)
        self.assertGreaterEqual(len(queue), 1)
        self.assertEqual(queue.iloc[0].screening_priority, "HIGH")
        self.assertEqual(diagnostic["high_priority_pairs"], 1)

    def test_strong_cross_original_pair_blocks_technical_gate(self):
        raw = self.fixture()
        i = int(raw.index[raw.original_split.eq("train")][0])
        j = int(raw.index[raw.original_split.eq("test")][0])
        pairs = pd.DataFrame([{"index_a": i, "index_b": j,
                               "source_split_a": "train", "source_split_b": "test",
                               "screening_priority": "HIGH"}])
        _, stats = build_split(raw, pairs, self.policy)
        self.assertFalse(stats["qa"]["no_high_priority_cross_original_source"])
        release = training_gate(stats, {"screen_incomplete_due_to_resource_budget": False,
                                      "complete_exhaustive_phash_search": True,
                                      "all_phash_candidates_image_scored": True}, self.policy)
        self.assertEqual(release["technical_split_gate"], "FAIL_NEEDS_INVESTIGATION")



class TestCompletePerceptualScreen(unittest.TestCase):
    def test_matches_independent_bruteforce(self):
        from src.data.split import exhaustive_phash_candidates
        rng = np.random.default_rng(2026)
        values = rng.integers(0, np.iinfo(np.uint64).max, 73, dtype=np.uint64)
        values[3] = values[8] ^ np.uint64(0b10101)
        values[9] = values[8] ^ np.uint64(0b111111)
        data = pd.DataFrame({"phash64": [f"{int(v):016x}" for v in values]})
        actual, info = exhaustive_phash_candidates(data, max_distance=6, block_size=7)
        observed = {(int(a), int(b), int(d)) for a,b,d in actual.itertuples(index=False, name=None)}
        expected = {(i,j,(int(values[i]) ^ int(values[j])).bit_count())
                    for i in range(len(values)) for j in range(i+1,len(values))
                    if (int(values[i]) ^ int(values[j])).bit_count() <= 6}
        self.assertEqual(observed, expected)
        self.assertTrue(info["complete_exhaustive_phash_search"])
        self.assertEqual(info["total_image_pairs_compared"], 73*72//2)

    def test_complete_search_many_identical_hashes_no_truncation(self):
        from src.data.split import exhaustive_phash_candidates
        df = pd.DataFrame({"phash64": ["0000000000000000"]*22})
        result, info = exhaustive_phash_candidates(df, block_size=4)
        self.assertEqual(len(result), 22*21//2)
        self.assertFalse(info["screen_incomplete_due_to_resource_budget"])

    def test_source_image_hash_mismatch_blocks_resume(self):
        import tempfile
        from PIL import Image
        from src.data.split import Source, load_candidate_thumbnails
        with tempfile.TemporaryDirectory() as tmp:
            p = Path(tmp) / "train" / "NORMAL"
            p.mkdir(parents=True)
            Image.new("L", (64,64), 80).save(p / "a.jpeg")
            df = pd.DataFrame({"relative_path": ["train/NORMAL/a.jpeg"],
                               "file_sha256": ["not_the_actual_sha"], "phash64": ["0000000000000000"]})
            pairs = pd.DataFrame([{"index_a":0,"index_b":0,"phash_hamming":0}])
            with self.assertRaises(RuntimeError):
                load_candidate_thumbnails(Source("folder",Path(tmp)),df,pairs)

    def test_no_auto_signoff_even_full_screen(self):
        from src.data.split import training_gate
        gate=training_gate({"qa":{"source_ok": True}},
            {"screen_incomplete_due_to_resource_budget": False,
             "complete_exhaustive_phash_search": True,
             "all_phash_candidates_image_scored": True},
            {"near_duplicate_screen": {"max_phash_distance": 6}})
        self.assertIn("HOLD", gate["research_training_gate"])
        self.assertEqual(gate["patient_independence"], "UNVERIFIED")

    def test_review_queue_keeps_high_and_review_only(self):
        from src.data.split import action_review_queue
        scored = pd.DataFrame({"screening_priority": ["LOW_SCREENED","HIGH","EXACT_KNOWN","REVIEW"]})
        out = action_review_queue(scored)
        self.assertEqual(list(out.screening_priority), ["HIGH","REVIEW"])


if __name__ == "__main__":
    unittest.main()

'''
TEST_PATH = PROJECT_ROOT / 'tests/test_split_integrity.py'
TEST_PATH.write_text(TEST_SOURCE)
run_tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
                           cwd=str(PROJECT_ROOT), capture_output=True, text=True)
unit_text = (run_tests.stdout or '') + '\n' + (run_tests.stderr or '')
(PROJECT_ROOT / 'reports/Phase 4/unit_test_results_v2.txt').write_text(unit_text)
print(unit_text[-6000:])
if run_tests.returncode:
    raise RuntimeError('Phase 4 V2 automated tests failed; refusing to continue.')


test_complete_search_many_identical_hashes_no_truncation (test_split_integrity.TestCompletePerceptualScreen.test_complete_search_many_identical_hashes_no_truncation) ... ok
test_matches_independent_bruteforce (test_split_integrity.TestCompletePerceptualScreen.test_matches_independent_bruteforce) ... ok
test_no_auto_signoff_even_full_screen (test_split_integrity.TestCompletePerceptualScreen.test_no_auto_signoff_even_full_screen) ... ok
test_review_queue_keeps_high_and_review_only (test_split_integrity.TestCompletePerceptualScreen.test_review_queue_keeps_high_and_review_only) ... ok
test_source_image_hash_mismatch_blocks_resume (test_split_integrity.TestCompletePerceptualScreen.test_source_image_hash_mismatch_blocks_resume) ... 
Checking only pHash candidate X-rays:   0%|          | 0/1 [00:00<?, ?image/s]
ok
test_candidate_split_reproducible (test_split_integrity.TestSplitIntegrity.test_candidate_split_reproducible) ... ok
test_conflicting_label_detected (test_split_integrity.TestSplit

### 5. Recover the verified dataset inventory without repeating the expensive scan
**Recommended:** Attach the old Phase 4 output ZIP as an additional Kaggle input. The notebook automatically searches under `/kaggle/input` and `/kaggle/working`, or use environment variable `PNEUMOLENS_PREVIOUS_PHASE4_ZIP` for an explicit path.

Resume is refused if the previous inventory SHA-256 does not match Phase 1, its previous run did not verify that fingerprint, or current image filenames disagree. When reusing prior hashes, the current files actually opened for similarity checking are rehashed individually; this is not described as a new complete source-byte audit.

In [5]:
from zipfile import ZipFile

def find_previous_phase4_zip():
    override = os.environ.get("PNEUMOLENS_PREVIOUS_PHASE4_ZIP", "").strip()
    if override:
        p = Path(override)
        if not p.is_file():
            raise FileNotFoundError(f"Explicit previous Phase 4 ZIP missing: {p}")
        return p
    name = "PneumoLens_Phase4_Split_Integrity_Outputs.zip"
    candidates = []
    for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
        if base.is_dir():
            candidates.extend(base.rglob(name))
    paths = sorted(set(p.resolve() for p in candidates if p.is_file()))
    if len(paths) > 1:
        raise RuntimeError("Multiple V1 output ZIPs found; set PNEUMOLENS_PREVIOUS_PHASE4_ZIP explicitly: " + str(paths[:6]))
    return paths[0] if paths else None

source = discover_source(search_root=os.environ.get("PNEUMOLENS_KAGGLE_INPUT", "/kaggle/input"))
previous_zip = find_previous_phase4_zip()
expected_files = {relative for _, _, relative in source.iter_images()}
assert len(expected_files) == sum(POLICY['source']['expected_counts'].values())
previous_manifest = None
prior_summary = {}
full_thumbnails = None

if previous_zip is not None:
    with ZipFile(previous_zip) as oldzip:
        required = ['data/processed/phase4_source_inventory_PRIVATE.csv',
                    'data/processed/phase4_candidate_split_manifest_PRIVATE.csv',
                    'reports/Phase 4/split_summary.json']
        absent = [x for x in required if x not in oldzip.namelist()]
        if absent:
            raise RuntimeError(f'Incomplete Phase4 V1 archive. Missing: {absent}')
        manifest = pd.read_csv(oldzip.open(required[0]), dtype={'phash64': 'string', 'file_sha256': 'string', 'decoded_sha256': 'string'})
        previous_manifest = pd.read_csv(oldzip.open(required[1]), dtype={'phash64': 'string'})
        prior_summary = json.loads(oldzip.read(required[2]).decode('utf-8'))
    manifest = manifest.sort_values('relative_path').reset_index(drop=True)
    if not prior_summary.get('phase1_dataset_fingerprint_verified', False):
        raise RuntimeError('The previous audit did not verify the source fingerprint.')
    if prior_summary.get('policy_version') != 'pneumolens-split-v1':
        raise RuntimeError('Unexpected V1 split policy; cannot trust resume inventory.')
    if len(manifest) != len(expected_files) or set(manifest.relative_path) != expected_files:
        raise RuntimeError('Current Kaggle image paths do not match prior verified inventory.')
    if inventory_fingerprint(manifest) != POLICY['source']['expected_dataset_file_sha256_fingerprint']:
        raise RuntimeError('Prior source inventory does not match Phase 1 SHA-256 reference.')
    if prior_summary.get('dataset_fingerprint') != inventory_fingerprint(manifest):
        raise RuntimeError('Prior public summary and private inventory disagree.')
    if not manifest.relative_path.is_unique or manifest.phash64.isna().any():
        raise RuntimeError('Cached manifest has missing hashes or duplicated paths.')
    if set(previous_manifest.relative_path) != expected_files:
        raise RuntimeError('Previous split manifest and source inventory paths disagree.')
    if split_fingerprint(previous_manifest.sort_values('relative_path').reset_index(drop=True)) != prior_summary.get('split_fingerprint'):
        raise RuntimeError('Cached split assignment does not match the signed V1 fingerprint.')
    source_mode = 'RESUME_V1_VERIFIED_INVENTORY'
    print('RESUME ENABLED: skipping full-source hashing. Old ZIP:', previous_zip)
else:
    print('Previous Phase 4 ZIP not mounted: scanning the images once with a progress indicator.')
    manifest, full_thumbnails = analyze_source(source)
    source_mode = 'FULL_SOURCE_RESCAN'

source_fingerprint = inventory_fingerprint(manifest)
if source_fingerprint != POLICY['source']['expected_dataset_file_sha256_fingerprint']:
    raise RuntimeError('Dataset fingerprint mismatch against Phase 1. STOP: wrong dataset copy.')
counts = {f'{split}/{label}': int(count) for (split,label),count in
          manifest.groupby(['original_split','label']).size().items()}
if counts != POLICY['source']['expected_counts']:
    raise RuntimeError(f'Dataset distribution differs from Phase 1: {counts}')
print('Source:', source.kind, source.location)
print('Resume mode:', source_mode, '| inventory SHA-256 matches Phase 1:', source_fingerprint)
print('Verified original class counts:', counts)

Previous Phase 4 ZIP not mounted: scanning the images once with a progress indicator.


Building source inventory (first-run fallback):   0%|          | 0/5856 [00:00<?, ?image/s]

Source: folder /kaggle/input/datasets/srabondcosta/pneumonia/ZhangLabData/CellData/chest_xray
Resume mode: FULL_SOURCE_RESCAN | inventory SHA-256 matches Phase 1: a7b36506fbccf5f77b5d80a73b7a53cca8515dba45ae959a8d0a62e5f219a87b
Verified original class counts: {'test/NORMAL': 234, 'test/PNEUMONIA': 390, 'train/NORMAL': 1349, 'train/PNEUMONIA': 3883}


### 6. Exhaustive perceptual screening — **zero truncation**
We compare every unordered pair of the **5,856** saved 64-bit pHashes in memory-bounded NumPy blocks. The number of pairs compared is recorded. This removes the V1 1.5-million-pair candidate-enumeration cap; it does **not** mean every possible image duplicate is detectable with pHash.

In [6]:
PHASH_LIMIT = int(POLICY['near_duplicate_screen']['max_phash_distance'])
all_hash_candidates, hash_diag = exhaustive_phash_candidates(
    manifest, max_distance=PHASH_LIMIT,
    block_size=int(POLICY['near_duplicate_screen']['chunk_size']))
assert hash_diag['total_image_pairs_compared'] == len(manifest) * (len(manifest)-1)//2
assert hash_diag['complete_exhaustive_phash_search']
print('All unique image pairs evaluated:', f"{hash_diag['total_image_pairs_compared']:,}")
print(f"pHash candidates (Hamming <= {PHASH_LIMIT}):", f"{len(all_hash_candidates):,}")
print('Distance distribution:', all_hash_candidates.phash_hamming.value_counts().sort_index().to_dict())

All unique image pairs evaluated: 17,143,440
pHash candidates (Hamming <= 6): 1,987
Distance distribution: {0: 31, 2: 16, 4: 193, 6: 1747}


### 7. Reopen only the candidate radiographs and check image content
A pHash proximity alone does **not** mean two images are the same. Each candidate image is decoded into a 96×96 thumbnail exactly once; its current file SHA-256 and pHash are checked against the previous inventory. Every shortlisted pair receives normalized pixel correlation, gradient correlation, normalized MAE, and aspect-ratio-difference measures.

This uses image pixels from the original test **only to audit possible duplicates**, not for selecting the validation fold or training thresholds.

In [7]:
thumb_cache = load_candidate_thumbnails(source, manifest, all_hash_candidates,
                                       full_thumbnails=full_thumbnails)
scored_pairs = score_phash_candidates(manifest, all_hash_candidates, thumb_cache, POLICY)
review_queue = action_review_queue(scored_pairs)
screen_diag = {**hash_diag,
    'phash_screened_pairs': len(scored_pairs),
    'all_phash_candidates_image_scored': len(scored_pairs) == len(all_hash_candidates),
    'unique_candidate_images_validated_against_saved_sha256': len(thumb_cache) if source_mode.startswith('RESUME') else len(thumb_cache),
    'review_queue_pairs': len(review_queue),
    'high_priority_pairs': int((review_queue.screening_priority == 'HIGH').sum()) if len(review_queue) else 0,
    'priority_counts': scored_pairs.screening_priority.value_counts().to_dict() if len(scored_pairs) else {},
    'source_mode': source_mode,
    'screen_incomplete_due_to_resource_budget': False,
}
if not screen_diag['all_phash_candidates_image_scored']:
    raise RuntimeError('Some candidates were not scored: stop and investigate.')
print('Image-content screening complete:', len(scored_pairs), '/', len(all_hash_candidates))
print('Priority categories:', screen_diag['priority_counts'])
print('Actionable HIGH / REVIEW pairs:', len(review_queue))
print('Exact known copies are counted but not re-labelled as new clinical duplicate discoveries.')

Image-content screening complete: 1987 / 1987
Priority categories: {'LOW_SCREENED': 1956, 'EXACT_KNOWN': 31}
Actionable HIGH / REVIEW pairs: 0
Exact known copies are counted but not re-labelled as new clinical duplicate discoveries.


### 8. Reconstruct and re-validate the grouped candidate train/validation split
V2 uses only original **training** images to choose a fold. HIGH similarity pairs entirely within original train are conservatively joined for grouping, without assuming patient identity. The original test is preserved byte-for-byte in membership and kept out of fold selection. No uncertain pair is silently removed.

In [8]:
final_manifest, split_diag = build_split(manifest, review_queue.copy(), POLICY)
repeat_manifest, _ = build_split(manifest, review_queue.copy(), POLICY)
repeatable = split_fingerprint(final_manifest) == split_fingerprint(repeat_manifest)
if not repeatable:
    raise RuntimeError('Deterministic split assignment did not reproduce.')
roles = final_manifest['split_role'].tolist()
scored_pairs['role_a'] = [roles[int(i)] for i in scored_pairs.index_a]
scored_pairs['role_b'] = [roles[int(i)] for i in scored_pairs.index_b]
scored_pairs['cross_original_split'] = (scored_pairs.source_split_a != scored_pairs.source_split_b)
scored_pairs['cross_final_role'] = scored_pairs.role_a != scored_pairs.role_b
scored_pairs['cross_final_evaluable_role'] = (
    scored_pairs.role_a.isin(['TRAIN','VALIDATION','TEST_LOCKED']) &
    scored_pairs.role_b.isin(['TRAIN','VALIDATION','TEST_LOCKED']) &
    scored_pairs.cross_final_role)
scored_pairs['cross_dev_train_validation'] = (
    ((scored_pairs.role_a == 'TRAIN') & (scored_pairs.role_b == 'VALIDATION')) |
    ((scored_pairs.role_b == 'TRAIN') & (scored_pairs.role_a == 'VALIDATION')))
scored_pairs['actionable'] = scored_pairs.screening_priority.isin(['HIGH','REVIEW'])
review_queue = scored_pairs.loc[scored_pairs.actionable].copy().reset_index(drop=True)
action_cross_source = int((scored_pairs.actionable & scored_pairs.cross_original_split).sum())
action_cross_roles = int((scored_pairs.actionable & scored_pairs.cross_final_evaluable_role).sum())
split_diag['qa']['no_actionable_cross_original_source'] = action_cross_source == 0
split_diag['qa']['no_actionable_cross_final_roles'] = action_cross_roles == 0
split_diag['qa']['complete_pHash_screen_and_image_scoring'] = bool(screen_diag['all_phash_candidates_image_scored'])
split_diag['qa']['prior_source_fingerprint_matches'] = source_fingerprint == POLICY['source']['expected_dataset_file_sha256_fingerprint']
split_diag['qa']['repeatable_split_assignment'] = repeatable
split_diag['qa']['no_exact_original_train_test_overlap'] = (
    split_diag['duplicate_diagnostics']['decoded_sha256_cross_original_split_groups'] == 0)
split_diag['qa']['no_known_exact_label_conflicts'] = (
    split_diag['duplicate_diagnostics']['decoded_sha256_conflicting_label_groups'] == 0)
split_diag['qa']['no_actionable_cross_candidate_partitions'] = action_cross_roles == 0
split_diag['qa']['pending_cross_partition_review_pairs'] = action_cross_roles
if len(review_queue):
    review_queue['cross_original_split'] = review_queue.source_split_a != review_queue.source_split_b
    review_queue['cross_final_evaluable_role'] = review_queue.cross_final_evaluable_role.astype(bool)
role_counts = final_manifest.split_role.value_counts().to_dict()
prior_changed = None
if previous_manifest is not None:
    from_previous = previous_manifest.set_index('relative_path').split_role
    prior_changed = int((final_manifest.set_index('relative_path').split_role != from_previous).sum())
print('Split role counts:', role_counts)
print('Validation fraction:', round(split_diag['validation_fraction_of_retained_train']*100, 3), '%')
print('Action-level cross-original pairs:', action_cross_source)
print('Action-level cross candidate-role pairs:', action_cross_roles)
print('V1 -> V2 role changes:', prior_changed if prior_changed is not None else 'no V1 comparison available')

Split role counts: {'TRAIN': 4166, 'VALIDATION': 1040, 'TEST_LOCKED': 624, 'EXCLUDED_EXACT_COPY': 26}
Validation fraction: 19.977 %
Action-level cross-original pairs: 0
Action-level cross candidate-role pairs: 0
V1 -> V2 role changes: no V1 comparison available


### 9. Decide the automated gate; never auto-authorize training
Passing means the prespecified automated checks completed, not that all images originate from distinct patients. Any action-level cross-role or cross-original match holds the technical gate for human adjudication, and **every** run retains research-training signoff as a separate manual decision.

In [9]:
release_gate = training_gate(split_diag, screen_diag, POLICY)
print('Technical gate:', release_gate['technical_split_gate'])
print('Research training gate:', release_gate['research_training_gate'])
print('Patient independence:', release_gate['patient_independence'])
failed = [k for k, v in split_diag['qa'].items() if isinstance(v,bool) and not v]
print('Failed automated QA checks:', failed or 'none')
print('Non-exact pHash candidates crossing train/validation:',
      int((scored_pairs.cross_dev_train_validation & (scored_pairs.screening_priority != 'EXACT_KNOWN')).sum()))
print('These counts include low-priority perceptual candidates; they are NOT confirmed duplicate counts.')

Technical gate: PASS_AUTOMATED_CHECKS_WITH_LIMITATIONS
Research training gate: HOLD_PENDING_HUMAN_REVIEW_AND_RISK_ACCEPTANCE
Patient independence: UNVERIFIED
Failed automated QA checks: none
Non-exact pHash candidates crossing train/validation: 487
These counts include low-priority perceptual candidates; they are NOT confirmed duplicate counts.


### 10. Save private candidate details and a small review gallery
Private outputs preserve all scored pair evidence, the candidate split, source hashes and a human-decision queue. The contact sheet includes actual radiographs and **must remain outside GitHub**. If no HIGH/REVIEW pairs exist, the gallery shows a few closest cross-partition pairs as a residual-risk spot check.

In [10]:
PRIVATE = PROJECT_ROOT / 'data/processed'
final_manifest.to_csv(PRIVATE/'phase4_candidate_split_manifest_V2_PRIVATE.csv',index=False)
manifest.to_csv(PRIVATE/'phase4_source_inventory_V2_PRIVATE.csv',index=False)
scored_pairs.to_csv(PRIVATE/'phase4_all_phash_candidates_SCREENED_PRIVATE.csv',index=False)
review_queue.to_csv(PRIVATE/'phase4_near_duplicate_review_queue_V2_PRIVATE.csv',index=False)
(PRIVATE/'phase4_split_index_V2_PRIVATE.json').write_text(json.dumps({
    'policy_version':POLICY['version'], 'source_fingerprint':source_fingerprint,
    'split_fingerprint':split_fingerprint(final_manifest),
    'patient_ids_verified':False, 'training_auto_approved':False,
    'role_counts':{k:int(v) for k,v in role_counts.items()},
},indent=2))

# Select informative pairs for a PRIVATE visual review sample.
examples = review_queue.copy()
if examples.empty:
    examples = scored_pairs[scored_pairs.cross_final_evaluable_role &
                            scored_pairs.screening_priority.ne('EXACT_KNOWN')].copy()
if examples.empty:
    examples = scored_pairs.copy()
if not examples.empty:
    examples['manual_first'] = (examples.source_split_a != examples.source_split_b).astype(int)*2 + examples.cross_final_evaluable_role.astype(int)
    examples = examples.sort_values(['manual_first','phash_hamming','pixel_corr'], ascending=[False,True,False])
    examples = examples.head(int(POLICY['near_duplicate_screen']['max_private_contact_pairs']))
canvas = Image.new('RGB',(1080, max(180, 180*max(len(examples),1))), '#ffffff')
draw = ImageDraw.Draw(canvas)
if examples.empty:
    draw.text((18,25),'No pHash candidates within the chosen threshold.',fill='#171717')
else:
    for row_num, r in enumerate(examples.itertuples(index=False)):
        y = row_num*180
        for part, idx in enumerate([int(r.index_a), int(r.index_b)]):
            tile = Image.fromarray(thumb_cache[idx]).resize((160,160),Image.Resampling.BILINEAR).convert('RGB')
            canvas.paste(tile,(12+part*182,y+8))
        txt = (f"{r.screening_priority} | pHash={int(r.phash_hamming)} | corr={float(r.pixel_corr):.3f}\n"
               f"{r.role_a} vs {r.role_b} | {r.source_split_a} vs {r.source_split_b}\n"
               f"{r.path_a[-55:]}\n{r.path_b[-55:]}")
        draw.multiline_text((395,y+25),txt,fill='#171717',spacing=9)
canvas.save(PRIVATE/'phase4_near_duplicate_contact_sheet_V2_PRIVATE.png')
print('Saved 5 private files under', PRIVATE)

Saved 5 private files under /kaggle/working/pneumolens_phase4_v2/project_files/data/processed


### 11. Public, aggregate-only charts and tables
No radiographs, source file paths, patient identifiers or individual hashes are included in the public report figures and aggregate summary tables.

In [11]:
FIG = PROJECT_ROOT/'reports/figures'
TAB = PROJECT_ROOT/'reports/tables'
plt.rcParams.update({'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
role_order=['TRAIN','VALIDATION','TEST_LOCKED','EXCLUDED_EXACT_COPY']
role_table=(final_manifest.groupby(['split_role','label']).size()
            .unstack(fill_value=0).reindex(role_order, fill_value=0))
role_table.reset_index().to_csv(TAB/'phase4v2_role_class_counts.csv',index=False)
ax=role_table[['NORMAL','PNEUMONIA']].plot.bar(stacked=True,figsize=(9,4.6),color=['#2a7fb8','#e99a48'],rot=0)
ax.set(title='PneumoLens | candidate split (V2)',xlabel='',ylabel='Radiographs (count)')
ax.legend(frameon=False)
ax.figure.tight_layout()
ax.figure.savefig(FIG/'phase4v2_01_split_class_distribution.png',dpi=160)
plt.close(ax.figure)

priority_table=(scored_pairs.groupby(['screening_priority','cross_original_split']).size()
                .unstack(fill_value=0) if len(scored_pairs) else pd.DataFrame())
if not priority_table.empty:
    priority_table=priority_table.rename(columns={False:'Same original partition',True:'Across original train/test'})
priority_table.to_csv(TAB/'phase4v2_similarity_priorities.csv')
if not priority_table.empty:
    ax=priority_table.plot.bar(stacked=True,figsize=(9,4.6),color=['#6b7280','#d97706'],rot=0)
    ax.set(title='Complete pHash screen: image-content evidence categories',xlabel='',ylabel='Pairs (not confirmed duplicates)')
    ax.legend(frameon=False)
    ax.figure.tight_layout()
    ax.figure.savefig(FIG/'phase4v2_02_similarity_screen.png',dpi=160)
    plt.close(ax.figure)

qa_table=pd.DataFrame([{'check':k,'passed':bool(v)} for k,v in split_diag['qa'].items() if isinstance(v,bool)])
qa_table.to_csv(TAB/'phase4v2_technical_checks.csv',index=False)
fig,ax=plt.subplots(figsize=(10, max(5,len(qa_table)*0.33)))
positions=np.arange(len(qa_table))
widths=np.where(qa_table.passed.to_numpy(),1.0,0.24)
ax.barh(positions,widths,color=['#15803d' if x else '#b91c1c' for x in qa_table.passed])
for j, passed in enumerate(qa_table.passed):
    if not passed:
        ax.text(0.12,j,'FAIL',color='white',fontsize=8,ha='center',va='center',fontweight='bold')
ax.set_yticks(positions,qa_table.check.str.replace('_',' ',regex=False))
ax.set_xlim(0,1.12); ax.set_xticks([0,1],['0','PASS']);ax.invert_yaxis()
ax.set_title('Phase 4 V2: automated integrity checks (not patient independence)')
fig.tight_layout();fig.savefig(FIG/'phase4v2_03_integrity_checks.png',dpi=160);plt.close(fig)
print('Public aggregate figures and tables exported.')

Public aggregate figures and tables exported.


### 12. Record a transparent research report, methods contract, and unsigned signoff template
The release decision is explicitly **manual**. A completed pHash screen is stronger evidence than the truncated V1 scan, but is **not** proof of absence of duplicate patients, correct anatomical localization, or generalization to another hospital.

In [12]:
RUN_UTC = datetime.now(timezone.utc).isoformat()
summary = {
    'project':'PneumoLens','phase':'Phase 4 V2 - leakage-aware split integrity',
    'run_utc':RUN_UTC,'policy_version':POLICY['version'],
    'source_mode':source_mode,'source_path':str(source.location),
    'source_fingerprint':source_fingerprint,'source_counts':counts,
    'previous_v1_split_fingerprint':prior_summary.get('split_fingerprint'),
    'v1_to_v2_split_role_changes':prior_changed,
    'policy_sha256':hashlib.sha256((PROJECT_ROOT/'configs/split_policy.json').read_bytes()).hexdigest(),
    'module_sha256':hashlib.sha256(MODULE_PATH.read_bytes()).hexdigest(),
    'split_fingerprint':split_fingerprint(final_manifest),
    'source_rescan_note': ('Resume reuses Phase 1-verified full file inventory and rehashes candidate source files; '
                           'it does not rehash all 5856 current source files.' if previous_zip is not None else
                           'All original source files were freshly hashed in this run.'),
    'original_test_untouched_for_selection':True,
    'split':split_diag,
    'similarity_screen':screen_diag,
    'candidate_cross_original_train_test':int(scored_pairs.cross_original_split.sum()) if len(scored_pairs) else 0,
    'candidate_cross_train_validation':int(scored_pairs.cross_dev_train_validation.sum()) if len(scored_pairs) else 0,
    'actionable_cross_original_train_test':action_cross_source,
    'actionable_cross_final_evaluable_roles':action_cross_roles,
    'automated_gate':release_gate,
    'patient_independence_verified':False,
    'model_training_automatically_authorized':False,
    'research_only':True,
    'runtime':RUNTIME,
}
PUB=PROJECT_ROOT/'reports/Phase 4'
(PUB/'split_summary_v2.json').write_text(json.dumps(summary,indent=2,default=str)+'\n')
qa_lines = '\n'.join('- {}: `{}`'.format('PASS' if bool(v) else 'FAIL', key) for key,v in split_diag['qa'].items() if isinstance(v,bool))
report=f'''# PneumoLens | Phase 4 V2 — completed similarity screen

**Executed:** {RUN_UTC}  
**Automated technical gate:** `{release_gate['technical_split_gate']}`  
**Research training gate:** `{release_gate['research_training_gate']}`  
**Patient independence:** `UNVERIFIED` (filename series ≠ verified patients)

## Data provenance
- Full inventory anchored to Phase 1 SHA-256: `{source_fingerprint}`.
- Source inspection mode: `{source_mode}`. {summary['source_rescan_note']}
- Original locked test images: {role_counts.get('TEST_LOCKED',0)}; unchanged and not used for model/fold selection.
- Exact redundant original-train images excluded: {role_counts.get('EXCLUDED_EXACT_COPY',0)}.
- Redundant original-test copies preserved: {split_diag['locked_test_redundant_exact_copies_preserved']}.

## Candidate split
- Training: **{role_counts.get('TRAIN',0)}**; validation: **{role_counts.get('VALIDATION',0)}**; test: **{role_counts.get('TEST_LOCKED',0)}**.
- Validation share: {split_diag['validation_fraction_of_retained_train']:.3%} of deduplicated original training data.
- Reproducible split SHA-256 fingerprint: `{summary['split_fingerprint']}`.
- Previously assigned rows moved in V2: {str(prior_changed)} (none if V1 comparison not mounted).

## Exhaustive pHash search and actual image-content checks
- All unique pHash comparisons completed: **{hash_diag['total_image_pairs_compared']:,}** (no LSH candidate limit).
- pHash Hamming distance <= {PHASH_LIMIT}: **{len(all_hash_candidates):,}** candidates.
- Pairs image-scored: **{len(scored_pairs):,} / {len(all_hash_candidates):,}**.
- Source train/test boundary candidates: **{summary['candidate_cross_original_train_test']:,}**.
- New train/validation boundary candidates: **{summary['candidate_cross_train_validation']:,}**.
- Priority counts: {screen_diag['priority_counts']}.
- HIGH/REVIEW image-content flags: **{len(review_queue):,}**, with **{action_cross_source:,}** crossing original source and **{action_cross_roles:,}** crossing final model-relevant partitions.

**Interpretation:** pHash similarity, even after pixel and gradient scoring, is a candidate-review signal — NOT a confirmed duplicate, lesion, shared patient, or clinical label. A low-priority score is NOT proof of independence. V2 fixes the resource-truncation bug without proving clinical generalization.

## Validation checks
{qa_lines}

## Next decision
Review all flagged HIGH/REVIEW images, with priority for any cross-original or cross-train/validation pairs. Use the private review CSV and contact sheet. Do not revise the locked test to improve outcomes. Complete `split_review_signoff_TEMPLATE.md` with documented disposition and limitations before independent training authorization. Do not claim patient-level independence without reliable patient IDs or external proof.

**Research prototype only; not for clinical diagnosis, triage, or treatment.**
'''
(PUB/'split_report_v2.md').write_text(report)
protocol='''# PneumoLens | Phase 4 V2 Methods Contract

1. Same Phase-1-fingerprinted Kermany / ZhangLab pediatric CXR source; original test images stay locked.
2. V1 exact duplicate handling is preserved: only redundant original-training copies are excluded, while original-test copies remain unchanged.
3. The 64-bit DCT pHash is compared **exhaustively** for all unordered image pairs in constant-size blocks. A pHash Hamming distance <=6 defines a candidate for **image-content screening**, not a duplicate.
4. Candidate radiographs are read only for integrity screening. Their raw SHA-256 and DCT pHash are confirmed against the cached source inventory when resuming. No test image is used to choose the validation fold or tune its parameters.
5. All pHash candidates receive 96x96 grayscale pixel correlation, gradient correlation, intensity MAE and aspect-delta checks using previously frozen V1 thresholds. HIGH/REVIEW are not confirmed duplicate labels.
6. All HIGH pairs entirely inside original training become conservative related-image grouping constraints. The validation fold is chosen only within deduplicated original training using fixed StratifiedGroupKFold seeds/thresholds. Any discovered cross-partition actionable candidate blocks the automated gate.
7. Because trusted patient identifiers are unavailable, filename-series groups are explicitly weak proxies. A passing technical gate is not a guarantee of patient-level independence; no automatic research-training signoff is granted.
8. Original test duplicate membership is maintained for reproducibility. In subsequent evaluation, report a separate locked-test duplicate sensitivity analysis without changing the official fixed test set.
9. Private source manifests and images are stored under data/processed/, excluded from GitHub. Public reports are aggregate-only.
10. To finalize: audit relevant image pairs, resolve concerns without selecting to maximize test outcomes, document limitations and sign the research-use review form.
'''
(PROJECT_ROOT/'docs/split_protocol_v2.md').write_text(protocol)
signoff='''# PneumoLens — Phase 4 V2 Human Review / Research Training Signoff

**Status:** UNSIGNED — NOT AN AUTOMATIC APPROVAL

- Dataset inventory fingerprint: ___________________
- Candidate split fingerprint: _____________________
- Full pHash-screen and raw-image review completed by: ______________
- Date and source evidence reference: ______________________________
- HIGH/REVIEW candidate dispositions (reference private review records): ___________________
- Original train/test independence limitations acknowledged: YES / NO
- Filename-series grouping *not* verified patient identity acknowledged: YES / NO
- Locked test untouched for fold selection: YES / NO
- Decision: [ ] HOLD  [ ] RESEARCH-ONLY TRAINING APPROVED WITH DOCUMENTED LIMITATIONS
- Reviewer, rationale, and signature: __________________________

*Approval here does not certify clinical safety, deployment readiness, or patient independence.*
'''
(PUB/'split_review_signoff_TEMPLATE.md').write_text(signoff)
print(report[:2400])

# PneumoLens | Phase 4 V2 — completed similarity screen

**Executed:** 2026-11-23T17:34:46.479512+00:00  
**Automated technical gate:** `PASS_AUTOMATED_CHECKS_WITH_LIMITATIONS`  
**Research training gate:** `HOLD_PENDING_HUMAN_REVIEW_AND_RISK_ACCEPTANCE`  
**Patient independence:** `UNVERIFIED` (filename series ≠ verified patients)

## Data provenance
- Full inventory anchored to Phase 1 SHA-256: `a7b36506fbccf5f77b5d80a73b7a53cca8515dba45ae959a8d0a62e5f219a87b`.
- Source inspection mode: `FULL_SOURCE_RESCAN`. All original source files were freshly hashed in this run.
- Original locked test images: 624; unchanged and not used for model/fold selection.
- Exact redundant original-train images excluded: 26.
- Redundant original-test copies preserved: 3.

## Candidate split
- Training: **4166**; validation: **1040**; test: **624**.
- Validation share: 19.977% of deduplicated original training data.
- Reproducible split SHA-256 fingerprint: `6185405f23d72cecf4dfe63f7ba8002db97a79a39be21f5c3

In [14]:

# ============================================================
# PneumoLens | Phase 4 V2
# Final Human Review Preparation & Research Safety Gate
# ============================================================

import json
from datetime import datetime, timezone
from IPython.display import display, FileLink

# 1. Collect cross-partition similarity candidates
candidates = scored_pairs.loc[
    scored_pairs["cross_original_split"].fillna(False).astype(bool)
    | scored_pairs["cross_dev_train_validation"].fillna(False).astype(bool)
].copy()

# 2. Prioritize suspicious pairs
priority_map = {
    "HIGH": 0,
    "REVIEW": 1,
    "LOW_SCREENED": 2,
    "EXACT_KNOWN": 3
}

candidates["manual_priority"] = (
    candidates["screening_priority"]
    .map(priority_map)
    .fillna(9)
    .astype(int)
)

candidates = candidates.sort_values(
    ["manual_priority", "phash_hamming",
     "pixel_corr", "gradient_corr"],
    ascending=[True, True, False, False],
    kind="stable"
)

# 3. Create a manual review worksheet
candidates.insert(
    0, "review_id",
    [f"PAIR-{i:04d}" for i in range(1, len(candidates) + 1)]
)

candidates["manual_decision"] = "NOT_REVIEWED"
candidates["reviewer"] = ""
candidates["reviewer_notes"] = ""
candidates["reviewed_at_utc"] = ""

review_file = (
    PRIVATE / "phase4_manual_review_worksheet_V2_PRIVATE.csv"
)

candidates.to_csv(review_file, index=False)

# 4. Save an honest research approval status
review_status = {
    "project": "PneumoLens",
    "phase": "Phase 4 V2",
    "record_created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "dataset_fingerprint": summary["source_fingerprint"],
    "split_fingerprint": summary["split_fingerprint"],
    "technical_gate": summary["automated_gate"][
        "technical_split_gate"
    ],
    "human_review_status": "PENDING",
    "training_approval":
        "HOLD_PENDING_HUMAN_REVIEW_AND_RISK_ACCEPTANCE",
    "patient_independence": "UNVERIFIED",
    "cross_original_candidates": int(
        scored_pairs["cross_original_split"].sum()
    ),
    "cross_train_validation_candidates": int(
        scored_pairs["cross_dev_train_validation"].sum()
    ),
    "automated_priority_flags": int(
        scored_pairs["screening_priority"]
        .isin(["HIGH", "REVIEW"]).sum()
    ),
    "manual_review_worksheet_rows": len(candidates),
    "research_only": True
}

status_file = PUB / "phase4_review_status.json"

status_file.write_text(
    json.dumps(review_status, indent=2) + "\n",
    encoding="utf-8"
)

# 5. Display results
print("=" * 60)
print("PNEUMOLENS | PHASE 4 V2 REVIEW PREPARATION")
print("=" * 60)
print("Automated technical gate:",
      review_status["technical_gate"])
print("Cross-boundary review candidates:", len(candidates))
print("Automated priority flags:",
      review_status["automated_priority_flags"])
print("Human review: PENDING")
print("Training approval: HOLD")
print("Patient independence: UNVERIFIED")
print("=" * 60)

print("\nManual review worksheet:")
display(FileLink(str(review_file)))

print("\nExisting visual review contact sheet:")
display(FileLink(str(
    PRIVATE / "phase4_near_duplicate_contact_sheet_V2_PRIVATE.png"
)))

print("\nResearch signoff template:")
display(FileLink(str(
    PUB / "split_review_signoff_TEMPLATE.md"
)))

print("\nPhase 4 review preparation completed.")


PNEUMOLENS | PHASE 4 V2 REVIEW PREPARATION
Automated technical gate: PASS_AUTOMATED_CHECKS_WITH_LIMITATIONS
Cross-boundary review candidates: 799
Automated priority flags: 0
Human review: PENDING
Training approval: HOLD
Patient independence: UNVERIFIED

Manual review worksheet:


/kaggle/working/pneumolens_phase4_v2/project_files/data/processed/phase4_manual_review_worksheet_V2_PRIVATE.csv


Existing visual review contact sheet:


/kaggle/working/pneumolens_phase4_v2/project_files/data/processed/phase4_near_duplicate_contact_sheet_V2_PRIVATE.png


Research signoff template:


/kaggle/working/pneumolens_phase4_v2/project_files/reports/Phase 4/split_review_signoff_TEMPLATE.md


Phase 4 review preparation completed.


### 13. Package the exact files into one Windows-project ZIP
The ZIP contains `src/`, `tests/`, `configs/`, `reports/`, `docs/`, and private `data/processed/` directories **relative to your local PneumoLens root**. It does not include the chest-X-ray dataset. Never upload the entire ZIP to public GitHub because it includes private image-level review outputs.

In [15]:
ZIP_PATH = WORK_ROOT.parent/'PneumoLens_Phase4_Split_Integrity_V2_Outputs.zip'
export_paths=[]
with ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for path in sorted(PROJECT_ROOT.rglob('*')):
        if not path.is_file() or path.suffix=='.pyc' or '__pycache__' in path.parts:
            continue
        relative=path.relative_to(PROJECT_ROOT).as_posix()
        z.write(path,arcname=relative)
        export_paths.append(relative)
with ZipFile(ZIP_PATH) as z:
    corrupt=z.testzip()
    if corrupt is not None:
        raise RuntimeError(f'ZIP verification failure: {corrupt}')
    assert sorted(z.namelist()) == sorted(export_paths)
print('Project-ready output ZIP:', ZIP_PATH)
print('Total files:', len(export_paths), '| compressed size:', round(ZIP_PATH.stat().st_size/1024/1024,2),'MiB')
for path in export_paths:
    print('   ',path)
display(FileLink(str(ZIP_PATH)))

Project-ready output ZIP: /kaggle/working/PneumoLens_Phase4_Split_Integrity_V2_Outputs.zip
Total files: 24 | compressed size: 2.24 MiB
    configs/split_policy.json
    data/processed/phase4_all_phash_candidates_SCREENED_PRIVATE.csv
    data/processed/phase4_candidate_split_manifest_V2_PRIVATE.csv
    data/processed/phase4_manual_review_worksheet_V2_PRIVATE.csv
    data/processed/phase4_near_duplicate_contact_sheet_V2_PRIVATE.png
    data/processed/phase4_near_duplicate_review_queue_V2_PRIVATE.csv
    data/processed/phase4_source_inventory_V2_PRIVATE.csv
    data/processed/phase4_split_index_V2_PRIVATE.json
    docs/split_protocol_v2.md
    reports/Phase 4/phase4_review_status.json
    reports/Phase 4/split_report_v2.md
    reports/Phase 4/split_review_signoff_TEMPLATE.md
    reports/Phase 4/split_summary_v2.json
    reports/Phase 4/unit_test_results_v2.txt
    reports/figures/phase4v2_01_split_class_distribution.png
    reports/figures/phase4v2_02_similarity_screen.png
    reports/fig

/kaggle/working/PneumoLens_Phase4_Split_Integrity_V2_Outputs.zip